In [4]:
def insert_patient_name(name, age):
    print(name)
    print(age)
    print('inserted into database')

In [6]:
insert_patient_name(name='Koyel', age=26)

Koyel
26
inserted into database


In [8]:
insert_patient_name(name='Koyel', age=-26)

Koyel
-26
inserted into database


In [ ]:
def insert_patient_name(name: str, age: int):
    print(name)
    print(age)
    print('inserted into database')

In [12]:
def insert_patient_name(name: str, age: int):
    if type(name) == str and type(age) == int and age>0:
        print(name)
        print(age)
        print('inserted into database')
    else:
        raise TypeError('Incorrect data type')

In [14]:
insert_patient_name(name='Koyel', age=-26)

TypeError: Incorrect data type

In [23]:
def insert_patient_data(name: str, age: int):
    if age<0:
        raise ValueError("Age can't be negative")
    if type(name) == str and type(age) == int:
        print(name)
        print(age)
        print('inserted into database')
    else:
        raise TypeError('incorrect data type')

# insert_patient_data('koyel', 25)

In [29]:
try:
    insert_patient_data('koyel', -25)
except ValueError as v:
    print(v)
except TypeError as t:
    print(t)
except Exception as e:
    print(e)

Age can't be negative


In [31]:
!pip install pydantic

In [69]:
!pip install pydantic[email]

   ---------------------------------------- 0.0/313.6 kB ? eta -:--:--
   ---------------------- ----------------- 174.1/313.6 kB 5.3 MB/s eta 0:00:01
   ---------------------------------------- 313.6/313.6 kB 4.8 MB/s eta 0:00:00


In [16]:
from pydantic import BaseModel

class Patient(BaseModel):
    name: str
    age: int

def insert_patient_data(name, age):
    print(name)
    print(age)
    print('inserted into database')

patient_info = {'name': 'Koyel', 'age': 26}

patient1 = Patient(**patient_info)
patient1

Patient(name='Koyel', age=26)

In [26]:
class Patient(BaseModel):
    name: str
    age: int
    # weight: float

def insert_patient_data(patient: Patient):
    print(patient.name)
    print(patient.age)
    print('inserted into database')

patient_info = {'name': 'Koyel', 'age': 26}

patient2 = Patient(**patient_info)
insert_patient_data(patient2)

Koyel
26
inserted into database


In [20]:
patient_info = {'name': 'Koyel', 'age': '26'}

patient2 = Patient(**patient_info)
insert_patient_data(patient2)

Koyel
26
inserted into database


In [22]:
patient_info = {'name': 'Koyel', 'age': 'twenty six'}

patient2 = Patient(**patient_info)
insert_patient_data(patient2)

ValidationError: 1 validation error for Patient
age
  Input should be a valid integer, unable to parse string as an integer [type=int_parsing, input_value='twenty six', input_type=str]
    For further information visit https://errors.pydantic.dev/2.5/v/int_parsing

In [40]:
from pydantic import BaseModel, StrictInt, EmailStr, AnyUrl, Field, field_validator, model_validator, computed_field
from typing import List, Dict, Optional, Annotated

class Patient(BaseModel):
    name: Optional[str] = Annotated[str, Field(default=None, max_length = 4, title='ab', description='cf', examples=['n', 'm'])]
    email: EmailStr
    linkedin_url: AnyUrl
    age: int # int will coerce string '34' also, StrictInt
    weight: float = Field(gt=0, lt=60)
    height: float
    married: bool = False
    allergies: List[str]
    contact_details: Dict[str, str]

    @field_validator('email')
    @classmethod
    def email_validator(cls, value):
        valid_domains= ['hdfc.com', 'icici.com']
        domain_name = value.split('@')[-1]
        if domain_name not in valid_domains:
            raise ValueError('not a valid domain')
        return value

    @field_validator('name')
    @classmethod
    def transform_name(cls, value):
        return value.upper()

    # field_validator does custom validation over single field
    @field_validator('age', mode = 'after')
    @classmethod
    def validate_age(cls, value):
        if 0<value<100:
            return value
        else:
            raise ValueError("Enter correct age.")

    # model_validation does custom validation over multiple fields together, over full pydantic model
    @model_validator(mode = 'after')
    @classmethod
    def validate_emergency_contact(cls, model):
        if model.age > 60 and 'emergency' not in model.contact_details:
            return ValueError("Patient older than 60 must have emergency number.")
        return model

    @computed_field
    @property
    def bmi(self) -> float:
        bmi = round(self.weight/(self.height**2), 2)
        return bmi
    


def insert_patient_data(patient: Patient):
    print(patient.name)
    print(patient.age)
    print(patient.married)
    print(patient.weight)
    print(patient.height)
    print(patient.bmi)
    print('inserted into database')

patient_info = {
    'name': 'koyel', 
    'age': '40', 
    'weight': 57, 
    'height' : 78,
    # 'married': True,
    'allergies': ['pollen', 'dust'], 
    'contact_details': {'email':'abc@gmail.com', 'phone': '123456'},
    'email':'abc@hdfc.com',
    'linkedin_url' : 'https://linkedin.com'
}
print(type(patient_info))
patient1 = Patient(**patient_info)
insert_patient_data(patient1)

<class 'dict'>
KOYEL
40
False
57.0
78.0
0.01
inserted into database


In [42]:
class Address(BaseModel):
    city: str
    state: str
    pin: int
    

class Patient(BaseModel):
    name: str
    gender: str = 'M'
    age: int
    address: Address
    # here as address we will take class Address as we knoe int, str all data types are also classes

address_dict = {'city': "A", 'state': "B", 'pin': 9}
address1 = Address(**address_dict)

pat_dict = {'name': "E", 'age': 1, 'address': address1}
pat1 = Patient(**pat_dict)

# serialization
temp = pat1.model_dump()
temp1 = pat1.model_dump(include = ['name', 'gender'])
temp2 = pat1.model_dump(exclude = ['name', 'gender',{'address':['state']}])
temp3 = pat1.model_dump(exclude_unset=True)
print(temp)
print(temp1)
print(temp2)
print(temp3)
print(pat1.name)
print(pat1.address.pin)

{'name': 'E', 'gender': 'M', 'age': 1, 'address': {'city': 'A', 'state': 'B', 'pin': 9}}
{'name': 'E', 'gender': 'M'}
{'age': 1, 'address': {'city': 'A', 'state': 'B', 'pin': 9}}
{'name': 'E', 'age': 1, 'address': {'city': 'A', 'state': 'B', 'pin': 9}}
E
9


In [50]:
temp = pat1.model_dump_json()
print(type(temp))
temp

<class 'str'>


'{"name":"E","gender":"M","age":1,"address":{"city":"A","state":"B","pin":9}}'